In [1]:
from config import CLASSES_TO_USE, ORGANISMS_TO_USE, PER_ORGANISM_PER_CLASS_ALLELES, RESULT_COLUMNS

#rewrite classes to use to only I class because it's only class I method
METHOD = 'NetMHCpan-4.1'
CLASSES_TO_USE = ['I']
ORGANISMS_TO_USE = ['Human']
ALLELES = [item  for organism_to_use in ORGANISMS_TO_USE 
                 for class_to_use in CLASSES_TO_USE 
                 for item in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]]
ALLELES

['HLA_A_02_01', 'HLA_B_27_05', 'HLA_A_03_01', 'HLA_B_07_02']

In [2]:
from mhctools import NetMHCpan4
from mhcnames import normalize_allele_name## ONLY FOR LINUX
import pandas as pd

In [3]:
# # Read datasets
DECOY_KEY = 'x1'

DATA_PATH = "/home/denklewer/Data/"
# INPUT_DATA_PATH = DATA_PATH + 'input/'
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + f'predictions/{METHOD}/'
# per_allele_val_df = {
#     allele_name :pd.read_csv(f"{INPUT_DATA_PATH}df_{allele_name}_{DECOY_KEY}_val_for_classification.csv", index_col=0) for allele_name in ALLELES
# }

In [4]:
from data_reading import read_genetech_data

per_allele_train_df, per_allele_val_df = read_genetech_data(DATA_PATH, CLASSES_TO_USE, ALLELES)



/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3361: DtypeWarning: Columns (6,7,8,9,14,21,22,23,24) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):


## Prediction code

In [5]:
per_allele_df = per_allele_val_df

In [6]:
TARGET_ALLELES = dict()

## HARDCODE allele renaming if needed
for allele_name in ALLELES:
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    else:
        target_allele_name = allele_name
    TARGET_ALLELES[allele_name] = (target_allele_name)


TARGET_ALLELES

{'HLA_A_02_01': 'HLA_A_02_01',
 'HLA_B_27_05': 'HLA_B_27_05',
 'HLA_A_03_01': 'HLA_A_03_01',
 'HLA_B_07_02': 'HLA_B_07_02'}

In [7]:
def splitByLengthAndFilter(fasta_dict, min_length = 9):
    unique_lengths = list(set(map(len, list(fasta_dict.values()))))
    unique_lengths = list(filter(lambda x: x>=min_length, unique_lengths ))
    per_length_dict_of_dicts = {target_length :{k: v for k, v in fasta_dict.items() if len(v) == target_length} for target_length in unique_lengths}
    return per_length_dict_of_dicts

def predictSequences(dict_of_dicts, allele_name):
    alleles = [normalize_allele_name(TARGET_ALLELES[allele_name])]   # normalize allele name here
    predictor = NetMHCpan4(
            alleles=alleles,
            program_name='netMHCpan-4.1')
    print(f"Now we are going to predict binders for  dataset {allele_name} using alleles parameter {alleles}")
    binding_predictions = [predictor.predict_subsequences(fasta_dict, 
                                                          peptide_lengths=[length]) # peek any element and take length
                           for length, fasta_dict in dict_of_dicts.items()]
    lengths = [len(prediction) for prediction in binding_predictions] 
    print(f"total processed peptides after filtering -  {sum(lengths)}")
    return binding_predictions


def predict_with_netmhc(per_allele_df):
    # add ID as column
    for allele_name in ALLELES:
        per_allele_df[allele_name][RESULT_COLUMNS['id']] =  'seq' + per_allele_df[allele_name].index.astype(str).values
    # make fasta-style dicts
    per_allele_fasta = {
        allele_name : {
            ID : PEPTIDE 
            for ID, PEPTIDE in zip('seq' + per_allele_df[allele_name].index.astype(str).values, per_allele_df[allele_name]['peptide'].values) 
        } 
        for allele_name in ALLELES 
    }
    
    # split by length
    per_allele_per_length_dict = dict() 
    for allele_name in ALLELES:
        fasta_dict = per_allele_fasta[allele_name]
        per_length_dict_of_dicts = splitByLengthAndFilter(fasta_dict)
        print(f"allele {allele_name} fasta splited into {len(per_length_dict_of_dicts)} dicts")
        per_allele_per_length_dict[allele_name] = per_length_dict_of_dicts

    # predict
    per_allele_predictions = dict()
    for allele_name, per_length_dict in per_allele_per_length_dict.items():
        predictions = predictSequences(per_length_dict, allele_name)
        per_allele_predictions[allele_name] = predictions
    # join lengths together
    per_allele_total_predictions = dict()
    for allele_name, predictions in per_allele_predictions.items():
        list_of_df = [prediction.to_dataframe() for prediction in predictions]
        df_result = pd.concat(list_of_df, ignore_index=True)
        per_allele_total_predictions[allele_name] = df_result
    return per_allele_total_predictions

In [8]:
%%time
per_allele_total_predictions = predict_with_netmhc(per_allele_df)

allele HLA_A_02_01 fasta splited into 6 dicts
allele HLA_B_27_05 fasta splited into 6 dicts
allele HLA_A_03_01 fasta splited into 6 dicts
allele HLA_B_07_02 fasta splited into 6 dicts
Now we are going to predict binders for  dataset HLA_A_02_01 using alleles parameter ['HLA-A*02:01']
total processed peptides after filtering -  59529
Now we are going to predict binders for  dataset HLA_B_27_05 using alleles parameter ['HLA-B*27:05']
total processed peptides after filtering -  145768
Now we are going to predict binders for  dataset HLA_A_03_01 using alleles parameter ['HLA-A*03:01']
total processed peptides after filtering -  15780
Now we are going to predict binders for  dataset HLA_B_07_02 using alleles parameter ['HLA-B*07:02']
total processed peptides after filtering -  28430
CPU times: user 8.38 s, sys: 822 ms, total: 9.2 s
Wall time: 1h 39min 59s


In [9]:
per_allele_total_predictions[ALLELES[0]]

,source_sequence_name,offset,peptide,allele,score,affinity,percentile_rank,prediction_method_name,length
0,seq101522,0,AAAAAAAQS,HLA-A*02:01,0.000103,40.875,0.050667,netMHCpan-4.1,9
1,seq72022,0,AAAAATQTI,HLA-A*02:01,0.013712,5.798,0.144155,netMHCpan-4.1,9
2,seq66108,0,AAAALHGYG,HLA-A*02:01,0.000006,80.000,0.039022,netMHCpan-4.1,9
3,seq103881,0,AAAIFTVQL,HLA-A*02:01,0.053955,2.809,0.209980,netMHCpan-4.1,9
4,seq87765,0,AAALKKKGW,HLA-A*02:01,0.000100,41.286,0.013403,netMHCpan-4.1,9
...,...,...,...,...,...,...,...,...,...
59524,seq58078,0,YYLTNQEKALTKFL,HLA-A*02:01,0.000021,63.750,0.096571,netMHCpan-4.1,14
59525,seq105567,0,YYMWKTTDRYVQQK,HLA-A*02:01,0.000008,76.667,0.054698,netMHCpan-4.1,14
59526,seq97561,0,YYNLRKPKEFTMNN,HLA-A*02:01,0.000001,95.000,0.015611,netMHCpan-4.1,14
59527,seq102448,0,YYNYHTYEDGSDSR,HLA-A*02:01,0.000008,76.667,0.021249,netMHCpan-4.1,14


### Join predictions for given allele

#### join by length

#### add binder label

In [10]:
for allele_name in per_allele_total_predictions.keys(): 
    df_result = per_allele_total_predictions[allele_name]
    per_allele_total_predictions[allele_name] = df_result.merge(per_allele_df[allele_name][[RESULT_COLUMNS['id'],RESULT_COLUMNS['label']]], on=['source_sequence_name'])
    
    
assert RESULT_COLUMNS['label'] in per_allele_total_predictions[ALLELES[0]].columns, f"label column must be in the result, this column is '{RESULT_COLUMNS['label']}'"
assert RESULT_COLUMNS['id'] in per_allele_total_predictions[ALLELES[0]].columns, f"id column must be in the result, this column is '{RESULT_COLUMNS['id']}'"
assert RESULT_COLUMNS['peptide'] in per_allele_total_predictions[ALLELES[0]].columns, f"peptide column must be in the result, this column is '{RESULT_COLUMNS['peptide']}'"

In [11]:
df = per_allele_total_predictions[ALLELES[0]]
df

,source_sequence_name,offset,peptide,allele,score,affinity,percentile_rank,prediction_method_name,length,binder
0,seq101522,0,AAAAAAAQS,HLA-A*02:01,0.000103,40.875,0.050667,netMHCpan-4.1,9,0.0
1,seq72022,0,AAAAATQTI,HLA-A*02:01,0.013712,5.798,0.144155,netMHCpan-4.1,9,0.0
2,seq66108,0,AAAALHGYG,HLA-A*02:01,0.000006,80.000,0.039022,netMHCpan-4.1,9,0.0
3,seq103881,0,AAAIFTVQL,HLA-A*02:01,0.053955,2.809,0.209980,netMHCpan-4.1,9,0.0
4,seq87765,0,AAALKKKGW,HLA-A*02:01,0.000100,41.286,0.013403,netMHCpan-4.1,9,0.0
...,...,...,...,...,...,...,...,...,...,...
59524,seq58078,0,YYLTNQEKALTKFL,HLA-A*02:01,0.000021,63.750,0.096571,netMHCpan-4.1,14,0.0
59525,seq105567,0,YYMWKTTDRYVQQK,HLA-A*02:01,0.000008,76.667,0.054698,netMHCpan-4.1,14,0.0
59526,seq97561,0,YYNLRKPKEFTMNN,HLA-A*02:01,0.000001,95.000,0.015611,netMHCpan-4.1,14,0.0
59527,seq102448,0,YYNYHTYEDGSDSR,HLA-A*02:01,0.000008,76.667,0.021249,netMHCpan-4.1,14,0.0


#### write to files

In [12]:
# for decoy_key in DECOY_PREFIXES:
for allele_name, total_df in per_allele_total_predictions.items():
    total_df.to_csv(f"{TARGET_PATH_TO_PREDICTIONS}{METHOD}_predictions_{allele_name}")